# FractionalAblationCAM: Step-by-step walkthrough

This notebook validates the FractionalAblationCAM pipeline by exercising each method in isolation and visualizing intermediate outputs:

- Load config, data, and a trained model
- Choose a target convolution layer
- Instantiate FractionalAblationCAM
- extract_activations_outputs: collect per-timestep activations and baseline scores
- _forward_with_channel_scaling: sanity-check score changes from scaling a single channel
- compute_channel_importance: compute per-channel importance curves and visualize drops
- rescale_cams: verify spatial resizing to input resolution
- compute_cam: produce per-timestep CAMs and overlay on frames

In [ ]:
# Environment and imports
import os
import torch
import snntorch as snn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from src.ablation_cam import FractionalAblationCAM
from src.utils.config_utils import setup_config_dictionary
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cam_time_series, combine_polarities, plot_aggregated_cam

# Repro + device
print(torch.seed())
torch.manual_seed(2329669718442129658)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = setup_config_dictionary()
config["DATASET_NAME"] = "tiny_imagenet"
config["MODEL_NAME"] = "vgg11"
config["NORM_TYPE"] = "bntt"

print('Device:', device)
print('Dataset:', config['DATASET_NAME'])
print('Model:', config['MODEL_NAME'])
print('Normalization Type:', config['NORM_TYPE'])

In [ ]:
# Load a tiny batch for inspection (batch size 1)
train_loader, test_loader = get_dataloaders(
    config['DATASET_NAME'],
    batch_size=1,
)

sample_events, sample_labels = next(iter(test_loader))
print('Raw sample shape from loader:', tuple(sample_events.shape))

# The models in this repo expect (B, T, C, H, W)
if sample_events.ndim == 4:  # (B, C, H, W) -> add time dim of 1
    sample_events = sample_events.unsqueeze(1)

B, T, C, H, W = sample_events.shape
print('Normalized shape (B,T,C,H,W):', (B, T, C, H, W))

num_channels = C

model = get_model(
    config['MODEL_NAME'],
    num_classes=len(train_loader.dataset.classes),
    num_channels=num_channels,
    time_steps=config['N_TIME_BINS'],
    beta=config['BETA'],
    norm_type=config['NORM_TYPE'],
).to(device)

interactive_load_model(model, config['MODEL_SAVE_DIR'], config['DATASET_NAME'])
model.eval()

sample_events = sample_events.to(device)
sample_label = sample_labels.to(device)
print('Label:', int(sample_label[0].item()))

## Pick a target convolutional layer

We enumerate convolutional layers and pick one near the end (or a specific index) to visualize feature maps. If your model/layer count differs, adjust the index below.

In [ ]:
# List conv layers and choose an index

target_type = snn.Leaky
# target_type = torch.nn.Conv2d

layers = [m for m in model.modules() if isinstance(m, target_type)]
print('Total layers:', len(layers))

# Heuristic: take a deeper layer if available, else last
layer_index = min(6, len(layers) - 1)
target_layer = layers[layer_index]
print('Using layer index:', layer_index, '->', target_layer)

cam = FractionalAblationCAM(model, target_layer, n_steps=8, channel_batch_size=16)
print('Fractions:', cam.fractions.cpu().numpy())

## Instantiate FractionalAblationCAM

We use n_steps to control the number of retention fractions and channel_batch_size to control parallelism when scanning channels.

## 1) extract_activations_outputs

Run a normal forward pass while hooking the target layer to collect per-timestep activations. We also collect baseline class scores.

In [ ]:
with torch.no_grad():
    activations, outputs = cam.extract_activations_outputs(sample_events)
print(f"Sample events shape (B,T,C,H,W): {tuple(sample_events.shape)}")
print('Activations shape (B,T,C,H,W):', tuple(activations.shape))
print('Outputs shape (B,T,num_classes):', tuple(outputs.shape))

# Visualize a few feature maps from the first time step (if C is large, subsample)
B, T, Cc, Hh, Ww = activations.shape
vis_t = min(9, T-1)
num_vis = min(6, Cc)

fig, axes = plt.subplots(1, num_vis, figsize=(num_vis*3, 3))
for i in range(num_vis):
    ax = axes[i]
    ax.imshow(activations[0, vis_t, i].cpu(), cmap='gray')
    ax.set_title(f'ch {i}')
    ax.axis('off')
plt.suptitle(f'Feature maps at t={vis_t}')
plt.tight_layout()
plt.show()

# Frame visualization for context
sample_frames = sample_events[0].detach().cpu()  # (T, C, H, W)
if sample_frames.shape[1] == 2:  # (T, 2, H, W) -> (T, 1, H, W)
    combined = combine_polarities(sample_frames.permute(0,2,3,1).numpy())  # (T,H,W)
    combined = np.expand_dims(combined, axis=1)
else: # 3 channels, assume RGB
    combined = sample_frames.numpy()
plot_frames(combined)

## 2) _forward_with_channel_scaling (sanity check)

We scale one channel at different fractions and confirm the class score reduces versus baseline. This validates the ablation hook.

In [ ]:
# Compute baseline mean score and predicted class
with torch.no_grad():
    base_out = model(sample_events)
base_mean = outputs.mean(dim=1)
pred = base_mean.argmax(dim=1)
pred_class = int(pred[0].item())
print('Predicted class:', pred_class)
print('Baseline score:', base_mean[0, pred_class].item())

# Scale a single channel across fractions
Cc = activations.shape[2]
probe_ch = min(0, Cc-1)
fractions = cam.fractions.to(sample_events.device)
G = len(fractions)

def run_scaled(ch):
    scaling = torch.ones(G, Cc, device=sample_events.device)
    scaling[:, ch] = fractions
    inputs_rep = sample_events.expand(G, *sample_events.shape[1:])
    with torch.no_grad():
        scores = cam._forward_with_channel_scaling(inputs_rep, scaling).mean(dim=1)[:, pred_class]
    return scores

scores = run_scaled(probe_ch)
plt.figure(figsize=(5,3))
plt.plot(fractions.cpu().numpy(), (base_mean[0, pred_class]-scores).cpu().numpy(), marker='o')
plt.xlabel('Retention fraction'); plt.ylabel('Score drop')
plt.title(f'Channel {probe_ch} ablation curve')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3) compute_channel_importance

Run the full importance computation with progress bars, then visualize per-channel drop curves and summarize top channels.

In [ ]:
with torch.no_grad():
    importance, acts2, drops_per_channel, fractions2 = cam.compute_channel_importance(
        sample_events, class_idx=
        pred_class
        #int(sample_label[0].item()) 
    )
    
print('importance:', tuple(importance.shape))
print('drops_per_channel:', tuple(drops_per_channel.shape))

# Visualize top-k channel sensitivity curves
k = min(6, importance.numel())
order = torch.argsort(importance, descending=True)
sel = order[:k]

plt.figure(figsize=(7,4))
for ch in sel:
    plt.plot(fractions2.cpu().numpy(), drops_per_channel[ch].cpu().numpy(), label=f'ch {int(ch)} (imp {importance[ch].item():.3f})')
plt.xlabel('Retention fraction'); plt.ylabel('Score drop (baseline - ablated)')
plt.title(f'Class {pred_class} sensitivity per channel')
plt.legend(fontsize=8, ncol=2); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Aggregate curve
plt.figure(figsize=(5,3))
mean_drop = drops_per_channel.mean(0).cpu().numpy()
min_drop = drops_per_channel.min(0).values.cpu().numpy()
max_drop = drops_per_channel.max(0).values.cpu().numpy()
plt.plot(fractions2.cpu().numpy(), mean_drop, color='black', label='Mean drop')
plt.fill_between(fractions2.cpu().numpy(), min_drop, max_drop, color='gray', alpha=0.2, label='Min-Max')
plt.xlabel('Retention fraction'); plt.ylabel('Score drop')
plt.title('Aggregate channel sensitivity'); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4) compute_cam

Use the computed importances to build per-timestep CAMs, then visualize as overlays and a time-series summary.

In [ ]:
cams = cam.compute_cam(sample_events, importance, normalize_per_timestep=False, apply_relu=True)
print('Per-timestep CAMs:', tuple(cams.shape))

print(activations.shape, cams.shape)

import pandas as pd
importance_df = pd.DataFrame(importance.cpu().numpy())
importance_df.sort_values(by=0, ascending=False, inplace=True)
print(importance_df.head(20))

# activations_df = pd.DataFrame(activations[0,0].cpu().numpy().reshape(activations.shape[2], -1))
# print(activations_df.head(20))
# cams_df = pd.DataFrame(cams[3].cpu().numpy())
# print(cams_df.head(20))

# Show overlays at a reasonable stride
T_total = cams.shape[0]
stride = max(1, T_total // 12)

# plot_cam_time_series expects frames as (T,C,H,W) or (1,T,C,H,W). We'll pass grayscale combined
cts = plot_cam_time_series(combined, cams, aggregate='mean', frame_stride=stride, per_frame_cam_norm=True)
cts.show()
ct_aggregate = plot_aggregated_cam(combined, cams, overlay_alpha=0.7)
ct_aggregate.show()

## Notes

- If the chosen conv layer index is out of range for your model, adjust `layer_index` above.
- For faster runs, reduce `n_steps` or `channel_batch_size`.
- To explain the ground-truth class instead of the predicted one, pass `class_idx=sample_label.item()` in `compute_channel_importance`.